In [1]:
# Imports
import polars as pl
from pathlib import Path
import os
os.chdir('/Users/michaelmayo/Downloads/data')

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "dat_train1.csv"
CLEAN_PATH = PROJECT_ROOT / "data" / "dat_train1_clean.csv"

df = pl.scan_csv(DATA_PATH)

In [2]:
## task 1

#1.1
num_rows = df.select(pl.len()).collect().item()
#1.2
num_unique_ids = df.select(pl.col("id").n_unique()).collect().item()

#1.3
earliest, latest = df.select([
    pl.col("event_timestamp").min().alias("earliest"),
    pl.col("event_timestamp").max().alias("latest")
]).collect().row(0)



print("Task 1.1 - Number of rows:", num_rows)

print("Task 1.2 - Number of unique IDs:", num_unique_ids)

print("Task 1.3 - Earliest timestamp:", earliest)
print("Task 1.3 - Latest timestamp:", latest)

Task 1.1 - Number of rows: 54960961
Task 1.2 - Number of unique IDs: 1430445
Task 1.3 - Earliest timestamp: 2020-11-03T03:31:30Z
Task 1.3 - Latest timestamp: 2023-01-23T12:29:56Z


In [3]:
import polars as pl

SUCCESS_EVENT = "order_shipped"

def create_journey_tables(input_csv_path):
    q = pl.scan_csv(input_csv_path)

    q = q.unique(subset=["id", "event_name", "event_timestamp"])

    q = q.with_columns([
        pl.col("event_timestamp").str.to_datetime(time_zone="UTC"),
        pl.col("event_name").cast(pl.Utf8),
        pl.col("ed_id").cast(pl.Int64),
    ])

    q = q.sort(["id", "event_timestamp", "ed_id"])

    max_time = q.select(pl.col("event_timestamp").max()).collect().item()

    journeys = (
        q.group_by("id")
        .agg([
            pl.struct(["event_timestamp", "event_name", "ed_id"]).alias("journey"),

            pl.len().alias("num_actions"),
            pl.col("ed_id").n_unique().alias("num_unique_actions"),

            pl.col("event_timestamp").min().alias("start_time"),
            pl.col("event_timestamp").max().alias("end_time"),

            (pl.col("event_timestamp").max() - pl.col("event_timestamp").min())
            .dt.total_seconds()
            .alias("duration_seconds"),

            pl.col("ed_id").first().alias("first_action"),
            pl.col("ed_id").last().alias("last_action"),

            pl.col("event_name").first().alias("first_event_name"),
            pl.col("event_name").last().alias("last_event_name"),

            pl.col("event_name").eq(SUCCESS_EVENT).any().alias("has_order_shipped"),

            pl.when(pl.col("event_name").eq(SUCCESS_EVENT))
            .then(pl.col("event_timestamp"))
            .otherwise(None)
            .min()
            .alias("completion_time"),
        ])
        .with_columns([
            ((pl.lit(max_time) - pl.col("end_time")) > pl.duration(days=60))
            .alias("older_than_60_days")
        ])
    ).collect()

    complete = journeys.filter(pl.col("has_order_shipped"))

    incomplete = journeys.filter(
        (~pl.col("has_order_shipped")) & (pl.col("older_than_60_days"))
    )

    ongoing = journeys.filter(
        (~pl.col("has_order_shipped")) & (~pl.col("older_than_60_days"))
    )

    return ongoing, incomplete, complete

In [4]:
training_csv_path = "~/Downloads/data/dat_train1.csv"

ongoing, incomplete, complete = create_journey_tables(training_csv_path)

print("ONGOING")
print(ongoing.head())

print("INCOMPLETE")
print(incomplete.head())

print("COMPLETE")
print(complete.head())

print("Counts:")
print("ongoing:", ongoing.height)
print("incomplete:", incomplete.height)
print("complete:", complete.height)

ONGOING
shape: (5, 14)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ id        ┆ journey   ┆ num_actio ┆ num_uniqu ┆ … ┆ last_even ┆ has_order ┆ completio ┆ older_th │
│ ---       ┆ ---       ┆ ns        ┆ e_actions ┆   ┆ t_name    ┆ _shipped  ┆ n_time    ┆ an_60_da │
│ str       ┆ list[stru ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ys       │
│           ┆ ct[3]]    ┆ u32       ┆ u32       ┆   ┆ str       ┆ bool      ┆ datetime[ ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆ μs, UTC]  ┆ bool     │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 209469390 ┆ [{2022-07 ┆ 181       ┆ 6         ┆ … ┆ browse_pr ┆ false     ┆ null      ┆ false    │
│ 0 1077740 ┆ -23       ┆           ┆           ┆   ┆ oducts    ┆           ┆           ┆          │
│ 162       ┆ 12:11:26  ┆           ┆           ┆   ┆           ┆   

In [5]:
import polars as pl
import random
import math
import numpy as np

def clean_col_name(x):
    return (
        str(x)
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
        .replace("(", "")
        .replace(")", "")
    )

def truncate_combined_table(combined_table, seed=42, max_samples_per_journey=60, top_n_events=50):
    random.seed(seed)

    # Top event names across all journeys
    all_event_names = []
    for row in combined_table.iter_rows(named=True):
        journey = row["journey"]
        if journey is not None:
            for e in journey:
                if e["event_name"] != SUCCESS_EVENT:
                    all_event_names.append(e["event_name"])

    top_events = (
        pl.DataFrame({"event_name": all_event_names})
        .group_by("event_name")
        .len()
        .sort("len", descending=True)
        .head(top_n_events)["event_name"]
        .to_list()
    )

    truncated_rows = []

    for row in combined_table.iter_rows(named=True):
        journey_id = row["id"]
        journey = row["journey"]
        start_time = row["start_time"]
        end_time = row["end_time"]
        source_table = row["source_table"]

        if journey is None or len(journey) == 0 or start_time is None or end_time is None:
            continue

        # Important: do not let complete journeys get too close to order_shipped
        if source_table == "complete" and row["completion_time"] is not None:
            effective_end_time = row["completion_time"]
            label = 1
        else:
            effective_end_time = end_time
            label = 0

        start_ts = start_time.timestamp()
        end_ts = effective_end_time.timestamp()

        if end_ts <= start_ts:
            continue

        duration_days = (end_ts - start_ts) / 86400
        n_samples = max(1, math.ceil(duration_days))
        n_samples = min(n_samples, max_samples_per_journey)

        event_times = [e["event_timestamp"] for e in journey]
        event_timestamps = [e["event_timestamp"].timestamp() for e in journey]
        event_ids = [e["ed_id"] for e in journey]
        event_names = [e["event_name"] for e in journey]

        for sample_num in range(1, n_samples + 1):

            if source_table == "complete":
                max_allowed_ts = start_ts + 0.70 * (end_ts - start_ts)
                cutoff_ts = random.uniform(start_ts, max_allowed_ts)
            else:
                cutoff_ts = random.uniform(start_ts, end_ts)

            keep_idx = 0
            for t in event_timestamps:
                if t <= cutoff_ts:
                    keep_idx += 1
                else:
                    break

            if keep_idx == 0:
                continue

            prefix_times = event_times[:keep_idx]
            prefix_ids = event_ids[:keep_idx]
            prefix_names = event_names[:keep_idx]

            # remove leakage
            non_leak = [
                i for i, name in enumerate(prefix_names)
                if name != SUCCESS_EVENT
            ]

            if len(non_leak) == 0:
                continue

            prefix_times = [prefix_times[i] for i in non_leak]
            prefix_ids = [prefix_ids[i] for i in non_leak]
            prefix_names = [prefix_names[i] for i in non_leak]

            first_time = prefix_times[0]
            last_time = prefix_times[-1]
            snapshot_time = last_time

            snapshot_age_days = (snapshot_time - start_time).total_seconds() / 86400
            active_span_days = (last_time - first_time).total_seconds() / 86400
            days_since_last_event = 0

            gaps = [
                (prefix_times[i] - prefix_times[i - 1]).total_seconds() / 86400
                for i in range(1, len(prefix_times))
            ]

            row_out = {
                "id": journey_id,
                "sample_num": sample_num,
                "source_table": source_table,
                "label": label,

                "snapshot_time": snapshot_time,
                "snapshot_age_days": snapshot_age_days,

                "num_events": len(prefix_names),
                "num_unique_event_names": len(set(prefix_names)),
                "num_unique_ed_ids": len(set(prefix_ids)),

                "first_event_time": first_time,
                "last_event_time": last_time,

                "first_event_name": prefix_names[0],
                "last_event_name": prefix_names[-1],

                "first_ed_id": prefix_ids[0],
                "last_ed_id": prefix_ids[-1],

                "active_span_days": active_span_days,
                "days_since_last_event": days_since_last_event,

                "events_per_day_since_start": len(prefix_names) / (snapshot_age_days + 1),
                "events_per_active_day": len(prefix_names) / (active_span_days + 1),

                "avg_gap_days": float(np.mean(gaps)) if len(gaps) > 0 else 0,
                "max_gap_days": float(np.max(gaps)) if len(gaps) > 0 else 0,
                "std_gap_days": float(np.std(gaps)) if len(gaps) > 0 else 0,

                "last1_event": prefix_names[-1],
                "last2_event": prefix_names[-2] if len(prefix_names) >= 2 else "missing",
                "last3_event": prefix_names[-3] if len(prefix_names) >= 3 else "missing",
                "last4_event": prefix_names[-4] if len(prefix_names) >= 4 else "missing",
                "last5_event": prefix_names[-5] if len(prefix_names) >= 5 else "missing",
            }

            for window in [1, 3, 7, 14, 30]:
                row_out[f"events_last_{window}d"] = sum(
                    (snapshot_time - t).total_seconds() / 86400 <= window
                    for t in prefix_times
                )

            row_out["recent_1d_to_30d"] = row_out["events_last_1d"] / (row_out["events_last_30d"] + 1)
            row_out["recent_3d_to_30d"] = row_out["events_last_3d"] / (row_out["events_last_30d"] + 1)
            row_out["recent_7d_to_30d"] = row_out["events_last_7d"] / (row_out["events_last_30d"] + 1)

            counts = {}
            for name in prefix_names:
                counts[name] = counts.get(name, 0) + 1

            for event in top_events:
                row_out[f"cnt_{clean_col_name(event)}"] = counts.get(event, 0)

            truncated_rows.append(row_out)

    return pl.DataFrame(truncated_rows)

In [6]:
# add a column telling us where each row came from
incomplete_labeled = incomplete.with_columns(
    pl.lit("incomplete").alias("source_table")
)

complete_labeled = complete.with_columns(
    pl.lit("complete").alias("source_table")
)

# combine
combined = pl.concat([incomplete_labeled, complete_labeled], how="vertical")

# practical move: sample first so it runs faster
combined_small = combined.sample(n=5000, shuffle=True, seed=42)

# truncate once
combined_truncate = truncate_combined_table(
    combined_small,
    seed=42,
    max_samples_per_journey=60
)

# split back apart
incomplete_truncate = combined_truncate.filter(
    pl.col("source_table") == "incomplete"
)

complete_truncate = combined_truncate.filter(
    pl.col("source_table") == "complete"
)

print("combined_small:", combined_small.height)
print("combined_truncate:", combined_truncate.height)
print("incomplete_truncate:", incomplete_truncate.height)
print("complete_truncate:", complete_truncate.height)

print(incomplete_truncate.head())
print(complete_truncate.head())

combined_small: 5000
combined_truncate: 212554
incomplete_truncate: 182393
complete_truncate: 30161
shape: (5, 59)
┌────────────┬────────────┬────────────┬───────┬───┬───────────┬───────────┬───────────┬───────────┐
│ id         ┆ sample_num ┆ source_tab ┆ label ┆ … ┆ cnt_accou ┆ cnt_catal ┆ cnt_appli ┆ cnt_finge │
│ ---        ┆ ---        ┆ le         ┆ ---   ┆   ┆ nt_downpa ┆ og_email_ ┆ cation_ph ┆ rhut_univ │
│ str        ┆ i64        ┆ ---        ┆ i64   ┆   ┆ ymentrece ┆ experian  ┆ one_decli ┆ ersity    │
│            ┆            ┆ str        ┆       ┆   ┆ ive…      ┆ ---       ┆ ned       ┆ ---       │
│            ┆            ┆            ┆       ┆   ┆ ---       ┆ i64       ┆ ---       ┆ i64       │
│            ┆            ┆            ┆       ┆   ┆ i64       ┆           ┆ i64       ┆           │
╞════════════╪════════════╪════════════╪═══════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ -155885386 ┆ 1          ┆ incomplete ┆ 0     ┆ … ┆ 0         ┆ 0         ┆ 

In [7]:
model_df = combined_truncate.fill_null(0)

model_df.write_csv("~/Downloads/data/model_training_realistic_from_truncation1.csv")

print(model_df.shape)
print(model_df["label"].value_counts())
print(model_df.head())

(212554, 59)
shape: (2, 2)
┌───────┬────────┐
│ label ┆ count  │
│ ---   ┆ ---    │
│ i64   ┆ u32    │
╞═══════╪════════╡
│ 0     ┆ 182393 │
│ 1     ┆ 30161  │
└───────┴────────┘
shape: (5, 59)
┌────────────┬────────────┬────────────┬───────┬───┬───────────┬───────────┬───────────┬───────────┐
│ id         ┆ sample_num ┆ source_tab ┆ label ┆ … ┆ cnt_accou ┆ cnt_catal ┆ cnt_appli ┆ cnt_finge │
│ ---        ┆ ---        ┆ le         ┆ ---   ┆   ┆ nt_downpa ┆ og_email_ ┆ cation_ph ┆ rhut_univ │
│ str        ┆ i64        ┆ ---        ┆ i64   ┆   ┆ ymentrece ┆ experian  ┆ one_decli ┆ ersity    │
│            ┆            ┆ str        ┆       ┆   ┆ ive…      ┆ ---       ┆ ned       ┆ ---       │
│            ┆            ┆            ┆       ┆   ┆ ---       ┆ i64       ┆ ---       ┆ i64       │
│            ┆            ┆            ┆       ┆   ┆ i64       ┆           ┆ i64       ┆           │
╞════════════╪════════════╪════════════╪═══════╪═══╪═══════════╪═══════════╪═══════════╪═══════════